# 🚕 NYC Taxi Trip Duration – Phần 4: XÂY DỰNG & ĐÁNH GIÁ MÔ HÌNH
### Tiêu chí (7) Regression Analysis — so sánh 5 mô hình bằng cross-validation

**Bài toán:** Hồi quy – dự đoán `trip_duration` (giây). Target huấn luyện: `y = log1p(trip_duration)`.
**Metric chính:** **RMSLE** = RMSE trên target đã `log1p` (metric của cuộc thi Kaggle).

### Sáu nguyên tắc so sánh công bằng — notebook này thực hiện thế nào
| # | Nguyên tắc | Thực hiện |
|---|---|---|
| 1 | Cùng dữ liệu, cùng đặc trưng, cùng cách chia | Mọi mô hình dùng chung `X_dev`, `y_dev`, cùng danh sách `FEATURES`, cùng một đối tượng `cv` |
| 2 | Mọi tiền xử lý nằm trong Pipeline | `ColumnTransformer` (impute + scale + one-hot) đặt trong Pipeline, fit lại trên train của **từng fold** |
| 3 | Chọn mô hình bằng CV, không bằng test | Mọi so sánh dựa trên CV; holdout chỉ dùng **một lần** ở Cell 13 |
| 4 | Hai tầng baseline | Baseline nội bộ: **Linear Regression** (+ `DummyRegressor` làm mốc tầm thường). Baseline bài báo: Cell 15 |
| 5 | Ít nhất hai metric | RMSLE (thang log) **kèm** MAE và RMSE theo **giây** |
| 6 | Tái lập được | `SEED = 42` ở ô đầu, `random_state` cho mọi mô hình, in phiên bản thư viện, lưu kết quả ra CSV |

### Nội dung
Cell 1–3 cấu hình & nạp dữ liệu · 4 loại bản ghi trùng · 5 cleaning · 6 lấy mẫu · 7 feature engineering ·
8 kiểm tra leakage/NaN · 9 chia 80/20 · 10 bộ tiền xử lý dùng chung · 11 định nghĩa mô hình & metric ·
12 **5-fold CV** · 13 đánh giá holdout một lần · 14 bảng kết quả (in đậm mô hình tốt nhất) ·
15 so với baseline bài báo · 16 kết luận & bước tiếp theo.

### ⚠️ Phạm vi bước này
**Chưa làm:** tinh chỉnh siêu tham số, SHAP, feature importance, ensemble, biểu đồ Dự đoán vs Thực tế / phần dư, lưu `best_model.joblib`, nộp bài Kaggle.
Theo hướng dẫn Mục 0, quy trình là **(1) CV 5 mô hình → (2) chọn 2–3 mô hình tốt nhất → (3) tinh chỉnh**. Notebook này dừng ở bước (1) và (2); các mục 7–11 của hướng dẫn sẽ nằm ở notebook tiếp theo.

## Cell 1 – Import, SEED & cấu hình
`SEED` đặt ở ô đầu và dùng lại cho mọi bước có yếu tố ngẫu nhiên (lấy mẫu, chia tập, CV, mô hình) — nguyên tắc 6 về khả năng tái lập.

In [1]:
import os
import glob
import json
import time
import random
import warnings

import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.makedirs('models', exist_ok=True)      # nơi lưu cv_results.csv, test_results.csv
os.makedirs('figures', exist_ok=True)

print('sklearn', sklearn.__version__, '| xgboost', xgboost.__version__,
      '| pandas', pd.__version__, '| numpy', np.__version__)

sklearn 1.6.1 | xgboost 3.2.0 | pandas 2.3.3 | numpy 2.0.2


## Cell 2 – Cấu hình chế độ chạy
`MODE` quyết định cỡ dữ liệu. Ba chế độ: `fast` kiểm tra pipeline, `baseline` là chế độ so sánh chính thức đưa vào báo cáo, `full` dùng toàn bộ dữ liệu sau cleaning.

**`USE_KNN`:** hướng dẫn liệt kê kNN trong bộ năm mô hình, nhưng Mục 12 của chính hướng dẫn cũng cảnh báo *kNN chạy rất lâu với hơn 30 nghìn dòng* (độ phức tạp dự đoán tỉ lệ với số dòng train × số dòng cần dự đoán). Với vài trăm nghìn dòng, riêng kNN có thể chiếm nhiều giờ. Mặc định để `False`; **bộ mô hình vẫn đủ 5** (Linear, Ridge, Random Forest, XGBoost, MLP) cộng thêm `DummyRegressor` làm mốc tầm thường. Bật `True` nếu muốn có kNN và chấp nhận thời gian chạy (nên bật kèm `MODE = 'fast'`).

In [2]:
MODE = 'fast'          # 'fast' (200k) | 'baseline' (300k – chính thức) | 'full'
SAMPLE_SIZE = {'fast': 200_000, 'baseline': 300_000, 'full': None}[MODE]
USE_KNN = False        # xem giải thích ở ô markdown phía trên
N_JOBS = -1

print(f'MODE = {MODE} | cỡ mẫu = {SAMPLE_SIZE if SAMPLE_SIZE else "toàn bộ sau cleaning"} | USE_KNN = {USE_KNN}')

MODE = fast | cỡ mẫu = 200000 | USE_KNN = False


## Cell 3 – Nạp dữ liệu
Tự tìm `train.csv` trong `/kaggle/input` (kể cả thư mục con). `dropoff_datetime` chỉ được đọc vào để kiểm tra trùng lặp, **không** dùng tạo feature.

In [3]:
SEARCH_DIRS = ['./data', '/kaggle/input', '.']

def find_file(name):
    for base in SEARCH_DIRS:
        if not os.path.isdir(base):
            continue
        for ext in ['.csv', '.zip', '.csv.zip']:
            hits = sorted(glob.glob(os.path.join(base, '**', name + ext), recursive=True))
            if hits:
                return hits[0]
    return None

train_path = find_file('train')
if train_path is None:
    raise FileNotFoundError('Không tìm thấy train.csv – kiểm tra dataset đã gắn vào notebook chưa!')

print('✅ Train:', train_path)
raw = pd.read_csv(train_path, parse_dates=['pickup_datetime'])
print(f'Dữ liệu gốc: {raw.shape[0]:,} dòng × {raw.shape[1]} cột')
raw.head(3)

✅ Train: /kaggle/input/competitions/nyc-taxi-trip-duration/train.zip
Dữ liệu gốc: 1,458,644 dòng × 11 cột


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124


## Cell 4 – Loại bản ghi trùng lặp hoàn toàn
Hai dòng giống nhau ở **mọi cột trừ `id`** gần như chắc chắn là một chuyến bị ghi hai lần. Nếu giữ, hai bản sao có thể rơi vào hai fold khác nhau → mô hình "xem trước đáp án" → **rò rỉ giữa các fold**, làm điểm CV đẹp hơn thực tế.
Chỉ loại bản sao **trùng khít**; các chuyến chỉ giống nhau ở vài cột là bản ghi hợp lệ và được giữ nguyên.

In [4]:
cols_no_id = [c for c in raw.columns if c != 'id']
dup_mask = raw.duplicated(subset=cols_no_id, keep=False)
n_nhom = raw.loc[dup_mask, cols_no_id].groupby(cols_no_id, dropna=False).ngroups if dup_mask.any() else 0
n_thua = int(raw.duplicated(subset=cols_no_id, keep='first').sum())

df = raw.drop_duplicates(subset=cols_no_id, keep='first').copy()
print(f'Số dòng dính trùng lặp hoàn toàn : {int(dup_mask.sum()):,}')
print(f'Số nhóm (chuyến bị ghi trùng)    : {n_nhom:,}')
print(f'Số bản sao thừa đã loại          : {n_thua:,} ({n_thua / len(raw):.5%})')
print(f'Còn lại: {len(df):,} dòng')

Số dòng dính trùng lặp hoàn toàn : 14
Số nhóm (chuyến bị ghi trùng)    : 7
Số bản sao thừa đã loại          : 7 (0.00048%)
Còn lại: 1,458,637 dòng


## Cell 5 – Cleaning (chỉ trên dữ liệu train)
Ngưỡng kế thừa nguyên vẹn từ Phần 1 để nhất quán giữa các bước. `speed_kmh` được tính **chỉ để lọc** rồi xoá ngay, vì nó chứa thông tin của target (rò rỉ nhãn).

> Các filter dựa trên target hoặc tốc độ **không bao giờ** áp cho tập test của Kaggle — test phải giữ đủ mọi chuyến.

In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    '''Khoảng cách đường chim bay (km) trên mặt cầu Trái Đất.'''
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


df['dist_km'] = haversine_km(df.pickup_latitude, df.pickup_longitude,
                             df.dropoff_latitude, df.dropoff_longitude)
df['speed_kmh'] = df['dist_km'] / (df['trip_duration'] / 3600)

RULES = {
    'trip_duration ngoài [60 giây, 21600 giây]': ~df.trip_duration.between(60, 21600),
    'passenger_count ngoài [1, 6]':              ~df.passenger_count.between(1, 6),
    'latitude ngoài [40.49, 40.92]':             ~(df.pickup_latitude.between(40.49, 40.92)
                                                   & df.dropoff_latitude.between(40.49, 40.92)),
    'longitude ngoài [-74.27, -73.68]':          ~(df.pickup_longitude.between(-74.27, -73.68)
                                                   & df.dropoff_longitude.between(-74.27, -73.68)),
    'dist_km < 0.05':                            df.dist_km < 0.05,
    'speed_kmh > 100':                           df.speed_kmh > 100,
}

n_truoc = len(df)
loai = pd.Series(False, index=df.index)
for ten, mask in RULES.items():
    mask = mask.fillna(True)
    print(f'{ten:<45}: {int(mask.sum()):>8,} dòng ({mask.mean():.4%})')
    loai |= mask

df = df[~loai].copy()
df.drop(columns=['speed_kmh'], inplace=True)          # xoá ngay để không thể lọt vào feature
print(f'\nTrước cleaning : {n_truoc:,} dòng')
print(f'Bị loại        : {n_truoc - len(df):,} dòng ({1 - len(df)/n_truoc:.4%})')
print(f'SỐ DÒNG CÒN LẠI: {len(df):,}')

trip_duration ngoài [60 giây, 21600 giây]    :   10,654 dòng (0.7304%)
passenger_count ngoài [1, 6]                 :       65 dòng (0.0045%)
latitude ngoài [40.49, 40.92]                :      642 dòng (0.0440%)
longitude ngoài [-74.27, -73.68]             :      702 dòng (0.0481%)
dist_km < 0.05                               :   11,328 dòng (0.7766%)
speed_kmh > 100                              :      175 dòng (0.0120%)

Trước cleaning : 1,458,637 dòng
Bị loại        : 17,886 dòng (1.2262%)
SỐ DÒNG CÒN LẠI: 1,440,751


## Cell 6 – Lấy mẫu cố định theo `MODE`
Lấy mẫu **sau cleaning, trước khi chia tập**, với `random_state=SEED`: cả năm mô hình dùng **cùng một mẫu** và **cùng các fold** — nguyên tắc 1 về so sánh công bằng.

In [6]:
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    print(f'Đã lấy mẫu cố định {len(df):,} dòng (MODE = {MODE})')
else:
    df = df.reset_index(drop=True)
    print(f'Dùng toàn bộ {len(df):,} dòng sau cleaning')

Đã lấy mẫu cố định 200,000 dòng (MODE = fast)


## Cell 7 – Feature engineering
Toàn bộ feature dưới đây là **phép biến đổi không học tham số** (chỉ phụ thuộc vào chính dòng đó), nên tính một lần trước khi chia tập là an toàn. Các bước **có** học tham số (impute, scale, one-hot) nằm trong Pipeline ở Cell 10 và được fit lại trong từng fold.

`hour_sin`/`hour_cos` mã hoá tính chu kỳ của giờ: 23h và 0h chỉ cách nhau 1 giờ chứ không phải 23.
PCA/K-Means **chưa dùng** ở bước baseline, để dành cho bước feature enhancement.

In [7]:
AIRPORTS = {'jfk': (40.6413, -73.7781), 'lga': (40.7769, -73.8740), 'ewr': (40.6895, -74.1745)}


def add_features(d):
    '''Tạo feature — chỉ gồm phép biến đổi không học tham số. Áp được cho cả train và test.'''
    d = d.copy()
    dt = d['pickup_datetime']
    plat, plon = d['pickup_latitude'], d['pickup_longitude']
    dlat, dlon = d['dropoff_latitude'], d['dropoff_longitude']

    # Thời gian
    d['hour'] = dt.dt.hour
    d['weekday'] = dt.dt.weekday
    d['month'] = dt.dt.month
    d['day_of_year'] = dt.dt.dayofyear
    d['week_hour'] = d['weekday'] * 24 + d['hour']
    d['is_weekend'] = (d['weekday'] >= 5).astype('int8')
    hf = d['hour'] + dt.dt.minute / 60
    d['hour_sin'] = np.sin(2 * np.pi * hf / 24)
    d['hour_cos'] = np.cos(2 * np.pi * hf / 24)

    # Khoảng cách & hướng
    d['dist_haversine'] = haversine_km(plat, plon, dlat, dlon)
    d['dist_manhattan'] = haversine_km(plat, plon, plat, dlon) + haversine_km(plat, plon, dlat, plon)
    d['log_dist'] = np.log1p(d['dist_haversine'])
    la1, lo1, la2, lo2 = map(np.radians, [plat, plon, dlat, dlon])
    d['bearing'] = np.degrees(np.arctan2(
        np.sin(lo2 - lo1) * np.cos(la2),
        np.cos(la1) * np.sin(la2) - np.sin(la1) * np.cos(la2) * np.cos(lo2 - lo1)))
    d['delta_lat'] = dlat - plat
    d['delta_lon'] = dlon - plon
    d['center_lat'] = (plat + dlat) / 2
    d['center_lon'] = (plon + dlon) / 2

    # Sân bay (bán kính 2 km)
    for ten, (alat, alon) in AIRPORTS.items():
        d[f'pickup_{ten}'] = (haversine_km(plat, plon, alat, alon) < 2).astype('int8')
        d[f'dropoff_{ten}'] = (haversine_km(dlat, dlon, alat, alon) < 2).astype('int8')
    d['is_airport_trip'] = d[[f'{p}_{a}' for p in ('pickup', 'dropoff') for a in AIRPORTS]].max(axis=1)

    d['store_and_fwd_flag'] = (d['store_and_fwd_flag'] == 'Y').astype('int8')
    return d


df = add_features(df)
print(f'Đã tạo feature. Dữ liệu: {df.shape[0]:,} dòng × {df.shape[1]} cột')

Đã tạo feature. Dữ liệu: 200,000 dòng × 35 cột


## Cell 8 – Chọn đặc trưng, kiểm tra rò rỉ & NaN/inf
`FEATURES` được liệt kê **tường minh** (thay vì `drop` cột thừa) nên không cột nào lọt vào ngoài ý muốn. Bốn cột cấm được `assert` kiểm tra bằng code.
Phân chia `NUM` / `CAT` theo đúng cột `vai_tro` trong Data Dictionary ở Phần 1: `vendor_id` là **biến phân loại** (số 2 không "lớn hơn" số 1) nên đưa vào `CAT` để one-hot.

In [8]:
CAT = ['vendor_id']                                   # biến phân loại → one-hot trong Pipeline
NUM = [
    'passenger_count', 'store_and_fwd_flag',
    'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
    'hour', 'weekday', 'month', 'day_of_year', 'week_hour', 'is_weekend', 'hour_sin', 'hour_cos',
    'dist_haversine', 'dist_manhattan', 'log_dist', 'bearing',
    'delta_lat', 'delta_lon', 'center_lat', 'center_lon',
    'pickup_jfk', 'dropoff_jfk', 'pickup_lga', 'dropoff_lga', 'pickup_ewr', 'dropoff_ewr',
    'is_airport_trip',
]
FEATURES = NUM + CAT
TARGET = 'log_trip_duration'
FORBIDDEN = ['dropoff_datetime', 'speed_kmh', 'trip_duration', 'id']

trung = [c for c in FORBIDDEN if c in FEATURES]
assert not trung, f'❌ Đặc trưng cấm lọt vào danh sách: {trung}'
print(f'✅ Không có đặc trưng cấm {FORBIDDEN}')
print(f'✅ NUM: {len(NUM)} cột | CAT: {len(CAT)} cột | Tổng: {len(FEATURES)} đặc trưng')

df[TARGET] = np.log1p(df['trip_duration'])
X = df[FEATURES].astype('float32')
y = df[TARGET].astype('float32')

n_nan, n_inf = int(X.isna().sum().sum()), int(np.isinf(X.to_numpy()).sum())
print(f'NaN trong X: {n_nan} | inf trong X: {n_inf} | NaN trong y: {int(y.isna().sum())}')
assert n_nan == 0 and n_inf == 0 and y.isna().sum() == 0, '❌ Có NaN/inf – phải xử lý trước khi train'
print(f'✅ Dữ liệu sạch. RAM của X: {X.memory_usage(deep=True).sum() / 1024**2:,.1f} MB')

✅ Không có đặc trưng cấm ['dropoff_datetime', 'speed_kmh', 'trip_duration', 'id']
✅ NUM: 29 cột | CAT: 1 cột | Tổng: 30 đặc trưng
NaN trong X: 0 | inf trong X: 0 | NaN trong y: 0
✅ Dữ liệu sạch. RAM của X: 22.9 MB


## Cell 9 – Chia 80% development / 20% final holdout
Cách chia ở Phần 1 là **ngẫu nhiên** (dữ liệu bảng, không có yếu tố nhóm) → CV tương ứng là `KFold(5, shuffle=True, random_state=SEED)` đúng theo bảng Mục 3.1 của hướng dẫn.

Holdout chỉ dùng **một lần** ở Cell 13, không tham gia CV, chọn đặc trưng hay tinh chỉnh.

> Ghi vào phần giới hạn của báo cáo: holdout đã qua cleaning nên "dễ" hơn tập test thật của Kaggle (test giữ nguyên mọi chuyến, kể cả ngoại lai) ⇒ **RMSLE holdout không so trực tiếp với điểm leaderboard**.

In [9]:
X_dev, X_hold, y_dev, y_hold = train_test_split(X, y, test_size=0.2, random_state=SEED)
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

print(f'Development (80%): {X_dev.shape[0]:,} dòng → dùng cho 5-fold CV')
print(f'Final holdout (20%): {X_hold.shape[0]:,} dòng → chỉ đánh giá 1 lần')

Development (80%): 160,000 dòng → dùng cho 5-fold CV
Final holdout (20%): 40,000 dòng → chỉ đánh giá 1 lần


## Cell 10 – Bộ tiền xử lý dùng chung
| Thành phần | Lý do |
|---|---|
| `SimpleImputer(median)` | Trung vị ít bị ảnh hưởng bởi ngoại lai. Dữ liệu hiện không có NaN, nhưng giữ trong Pipeline để an toàn khi áp lên dữ liệu mới |
| `StandardScaler` | Bắt buộc với mô hình tuyến tính có điều chuẩn, kNN và MLP; **không ảnh hưởng** mô hình cây nên dùng chung cho cả năm mô hình để so sánh công bằng (nguyên tắc 1) |
| `OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20)` | Test có thể chứa nhóm chưa gặp ở train; `min_frequency` gộp nhóm hiếm, tránh bùng nổ số cột |

Toàn bộ nằm trong `ColumnTransformer` đặt trong Pipeline ⇒ **fit lại trên phần train của từng fold**, không bao giờ nhìn thấy fold kiểm tra (nguyên tắc 2).

In [10]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('oh', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                          min_frequency=20, sparse_output=False))])
pre = ColumnTransformer([('num', num_pipe, NUM), ('cat', cat_pipe, CAT)])
print(pre)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imp',
                                                  SimpleImputer(strategy='median')),
                                                 ('sc', StandardScaler())]),
                                 ['passenger_count', 'store_and_fwd_flag',
                                  'pickup_longitude', 'pickup_latitude',
                                  'dropoff_longitude', 'dropoff_latitude',
                                  'hour', 'weekday', 'month', 'day_of_year',
                                  'week_hour', 'is_weekend', 'hour_sin',
                                  'hour_cos', 'dist_haversine',
                                  'dist_man...st', 'bearing',
                                  'delta_lat', 'delta_lon', 'center_lat',
                                  'center_lon', 'pickup_jfk', 'dropoff_jfk',
                                  'pickup_lga', 'dropoff_lga', 'pickup_ewr',
              

## Cell 11 – Bộ mô hình và metric

| Mô hình | Vai trò | Lý do đưa vào (viết vào báo cáo) |
|---|---|---|
| `DummyRegressor` | Mốc tầm thường | Luôn đoán giá trị trung bình; mô hình nào không vượt được thì vô nghĩa |
| **Linear Regression** | **Baseline nội bộ** | Đơn giản, hệ số dễ giải thích; là mô hình thường dùng trong các bài báo |
| Ridge | Tuyến tính có điều chuẩn | Ổn định hơn khi đặc trưng đa cộng tuyến (thấy ở heatmap tương quan Phần 3) |
| kNN *(tuỳ chọn)* | Dựa trên khoảng cách | Bắt quan hệ cục bộ; cho thấy vai trò của chuẩn hoá. Rất chậm với dữ liệu lớn |
| Random Forest | Cây, bagging | Bắt quan hệ phi tuyến và tương tác, ít cần tinh chỉnh |
| XGBoost | Cây tăng cường | Thường tốt nhất trên dữ liệu bảng |
| MLP | Mạng nơ-ron (mô hình thứ sáu, tuỳ chọn theo hướng dẫn) | Kiểm tra xem mạng nơ-ron có lợi thế gì trên dữ liệu này không |

**Không early stopping ở XGBoost:** dùng `n_estimators` cố định để công bằng với các mô hình khác. Early stopping để dành cho bước tinh chỉnh.
Riêng MLP bật `early_stopping=True` vì cơ chế này tách 10% **ngay trong phần train của fold**, không đụng tới fold kiểm tra hay holdout.

> Nếu MLP xếp cuối bảng, nguyên nhân gần như luôn là **chưa hội tụ** với `max_iter` mặc định, chứ không phải mạng nơ-ron không phù hợp. Ghi nhận đúng như vậy và để việc tăng `max_iter` cho bước tinh chỉnh.

In [11]:
models = {
    'Dummy (mean)':      DummyRegressor(strategy='mean'),
    'Linear (baseline)': LinearRegression(),
    'Ridge':             Ridge(alpha=1.0, random_state=SEED),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=10,
                                               max_features=0.5, n_jobs=N_JOBS, random_state=SEED),
    'XGBoost':           XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=8,
                                      min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                                      reg_lambda=1.0, tree_method='hist',
                                      n_jobs=N_JOBS, random_state=SEED),
    'MLP':               MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu',
                                      batch_size=1024, learning_rate_init=1e-3, max_iter=60,
                                      early_stopping=True, n_iter_no_change=5, random_state=SEED),
}
if USE_KNN:
    models['kNN'] = KNeighborsRegressor(n_neighbors=10, n_jobs=N_JOBS)

print(f'Số mô hình: {len(models)} → ' + ', '.join(models))
print('Baseline nội bộ: Linear (baseline) | Mốc tầm thường: Dummy (mean)')

Số mô hình: 6 → Dummy (mean), Linear (baseline), Ridge, Random Forest, XGBoost, MLP
Baseline nội bộ: Linear (baseline) | Mốc tầm thường: Dummy (mean)


In [12]:
def to_seconds(y_log):
    '''Đổi target từ thang log về giây; chặn giá trị âm do sai số dự đoán.'''
    return np.clip(np.expm1(np.asarray(y_log, dtype=np.float64)), 0, None)


# ---- Metric THANG LOG ----
def rmsle(y_true_log, y_pred_log):            # RMSE trên log1p ⇒ chính là RMSLE
    return np.sqrt(mean_squared_error(y_true_log, y_pred_log))

def mae_log(y_true_log, y_pred_log):
    return mean_absolute_error(y_true_log, y_pred_log)

# ---- Metric THANG GIÂY (đơn vị thật) ----
def rmse_seconds(y_true_log, y_pred_log):
    return np.sqrt(mean_squared_error(to_seconds(y_true_log), to_seconds(y_pred_log)))

def mae_seconds(y_true_log, y_pred_log):
    return mean_absolute_error(to_seconds(y_true_log), to_seconds(y_pred_log))


# scikit-learn luôn TỐI ĐA hoá điểm ⇒ metric lỗi phải mang dấu âm (greater_is_better=False)
SCORING = {
    'rmsle':        make_scorer(rmsle, greater_is_better=False),          # THANG LOG – metric chính
    'mae_log':      make_scorer(mae_log, greater_is_better=False),        # THANG LOG
    'mae_seconds':  make_scorer(mae_seconds, greater_is_better=False),    # THANG GIÂY
    'rmse_seconds': make_scorer(rmse_seconds, greater_is_better=False),   # THANG GIÂY
    'r2':           'r2',                                                 # THANG LOG
}
MAIN, BETTER_HIGH = 'rmsle', False        # metric chính và chiều "tốt hơn"

print('THANG LOG  : rmsle (chính), mae_log, r2')
print('THANG GIÂY : mae_seconds, rmse_seconds (tính từ expm1 của từng fold)')

THANG LOG  : rmsle (chính), mae_log, r2
THANG GIÂY : mae_seconds, rmse_seconds (tính từ expm1 của từng fold)


## Cell 12 – 5-fold Cross-Validation trên 80% development set
Hàm `so_sanh_cv` chạy CV cho từng mô hình và trả về **trung bình, độ lệch chuẩn và điểm train** của mỗi metric.
Cột `_train` tốt hơn hẳn cột `_mean` là dấu hiệu **quá khớp** — cần nêu ở phần giới hạn.
`n_jobs=1` ở `cross_validate` vì Random Forest và XGBoost đã tự chạy song song; song song hai tầng sẽ tranh CPU và nhân đôi RAM.

In [13]:
def so_sanh_cv(models, X, y, scoring, cv, pre, groups=None):
    '''Chạy CV cho từng mô hình; trả về bảng trung bình, độ lệch chuẩn và điểm train của mỗi metric.'''
    rows = []
    for name, model in models.items():
        pipe = model if isinstance(model, Pipeline) else Pipeline([('pre', pre), ('model', model)])
        t0 = time.time()
        r = cross_validate(pipe, X, y, cv=cv, scoring=scoring, groups=groups,
                           return_train_score=True, n_jobs=1, error_score='raise')
        row = {'mo_hinh': name}
        for k in scoring:
            # Mọi metric lỗi đều dùng greater_is_better=False ⇒ sklearn trả về giá trị ÂM,
            # phải đổi dấu để về giá trị thật. Riêng r2 giữ nguyên.
            dau = 1 if k == 'r2' else -1
            row[f'{k}_mean'] = dau * r[f'test_{k}'].mean()
            row[f'{k}_std'] = r[f'test_{k}'].std()
            row[f'{k}_train'] = dau * r[f'train_{k}'].mean()
        row['giay'] = round(time.time() - t0, 1)
        rows.append(row)
        print(f'xong: {name:<18} {MAIN} CV = {row[f"{MAIN}_mean"]:.4f} ± {row[f"{MAIN}_std"]:.4f} '
              f'| train = {row[f"{MAIN}_train"]:.4f} | {row["giay"]}s')
    return pd.DataFrame(rows)


cv_table = so_sanh_cv(models, X_dev, y_dev, SCORING, cv, pre)
cv_table.sort_values(f'{MAIN}_mean', ascending=not BETTER_HIGH).round(4)

xong: Dummy (mean)       rmsle CV = 0.7274 ± 0.0016 | train = 0.7274 | 2.9s
xong: Linear (baseline)  rmsle CV = 0.4035 ± 0.0026 | train = 0.4034 | 3.9s
xong: Ridge              rmsle CV = 0.4046 ± 0.0027 | train = 0.4045 | 3.4s
xong: Random Forest      rmsle CV = 0.3297 ± 0.0023 | train = 0.2643 | 227.5s
xong: XGBoost            rmsle CV = 0.3071 ± 0.0015 | train = 0.2481 | 39.6s
xong: MLP                rmsle CV = 0.3459 ± 0.0019 | train = 0.3414 | 112.5s


,mo_hinh,rmsle_mean,rmsle_std,rmsle_train,mae_log_mean,mae_log_std,mae_log_train,mae_seconds_mean,mae_seconds_std,mae_seconds_train,rmse_seconds_mean,rmse_seconds_std,rmse_seconds_train,r2_mean,r2_std,r2_train,giay
4,XGBoost,0.3071,0.0015,0.2481,0.2236,0.0008,0.1852,167.9226,0.9319,143.3085,295.6940,14.9154,258.9010,0.8217,0.0022,0.8836,39.6
3,Random Forest,0.3297,0.0023,0.2643,0.2436,0.0010,0.1929,183.1349,1.2064,147.6087,316.0970,14.5224,271.1942,0.7945,0.0033,0.8680,227.5
5,MLP,0.3459,0.0019,0.3414,0.2564,0.0014,0.2533,195.6624,2.0236,192.4314,342.2116,19.3707,329.6782,0.7738,0.0033,0.7797,112.5
1,Linear (baseline),0.4035,0.0026,0.4034,0.3078,0.0019,0.3077,235.2061,1.0902,235.1456,385.2120,12.4919,385.2692,0.6923,0.0043,0.6925,3.9
2,Ridge,0.4046,0.0027,0.4045,0.3088,0.0019,0.3088,235.8203,1.1363,235.7819,385.7956,12.6100,385.9003,0.6906,0.0046,0.6907,3.4
0,Dummy (mean),0.7274,0.0016,0.7274,0.5815,0.0009,0.5815,443.0016,1.5313,443.0011,689.8517,8.1204,689.8959,-0.0000,0.0000,0.0000,2.9


In [14]:
# Bảng gọn để đọc nhanh: metric chính + hai metric thang giây + dấu hiệu quá khớp
goc = cv_table.set_index('mo_hinh')
bang_doc = pd.DataFrame({
    'RMSLE_CV (log)': goc['rmsle_mean'],
    '± std': goc['rmsle_std'],
    'RMSLE_train (log)': goc['rmsle_train'],
    'Chênh train-CV': goc['rmsle_mean'] - goc['rmsle_train'],
    'MAE_CV (giây)': goc['mae_seconds_mean'],
    'RMSE_CV (giây)': goc['rmse_seconds_mean'],
    'R²_CV (log)': goc['r2_mean'],
    'Thời gian (s)': goc['giay'],
}).sort_values('RMSLE_CV (log)')
display(bang_doc.round(4))

print('Chênh train-CV càng lớn ⇒ càng có dấu hiệu quá khớp (mô hình thuộc dữ liệu train).')

,RMSLE_CV (log),± std,RMSLE_train (log),Chênh train-CV,MAE_CV (giây),RMSE_CV (giây),R²_CV (log),Thời gian (s)
mo_hinh,,,,,,,,
XGBoost,0.3071,0.0015,0.2481,0.0590,167.9226,295.6940,0.8217,39.6
Random Forest,0.3297,0.0023,0.2643,0.0655,183.1349,316.0970,0.7945,227.5
MLP,0.3459,0.0019,0.3414,0.0046,195.6624,342.2116,0.7738,112.5
Linear (baseline),0.4035,0.0026,0.4034,0.0001,235.2061,385.2120,0.6923,3.9
Ridge,0.4046,0.0027,0.4045,0.0001,235.8203,385.7956,0.6906,3.4
Dummy (mean),0.7274,0.0016,0.7274,-0.0000,443.0016,689.8517,-0.0000,2.9


Chênh train-CV càng lớn ⇒ càng có dấu hiệu quá khớp (mô hình thuộc dữ liệu train).


In [15]:
# Quy tắc đọc bảng của hướng dẫn: chênh lệch giữa 2 mô hình nhỏ hơn độ lệch chuẩn giữa các fold
# ⇒ coi như ngang nhau, ưu tiên mô hình đơn giản/nhanh hơn.
xep_hang = cv_table.sort_values(f'{MAIN}_mean', ascending=not BETTER_HIGH).reset_index(drop=True)
nhat, nhi = xep_hang.iloc[0], xep_hang.iloc[1]
chenh = abs(nhat[f'{MAIN}_mean'] - nhi[f'{MAIN}_mean'])

print(f'Hạng 1: {nhat.mo_hinh} = {nhat[f"{MAIN}_mean"]:.4f} ± {nhat[f"{MAIN}_std"]:.4f} ({nhat.giay}s)')
print(f'Hạng 2: {nhi.mo_hinh} = {nhi[f"{MAIN}_mean"]:.4f} ± {nhi[f"{MAIN}_std"]:.4f} ({nhi.giay}s)')
print(f'Chênh lệch = {chenh:.4f} | std của hạng 1 = {nhat[f"{MAIN}_std"]:.4f}')
if chenh < nhat[f'{MAIN}_std']:
    nhanh = nhat if nhat.giay <= nhi.giay else nhi
    print(f'→ Chênh lệch NHỎ HƠN độ lệch chuẩn giữa các fold ⇒ coi như ngang nhau; '
          f'ưu tiên mô hình nhanh hơn: {nhanh.mo_hinh}')
else:
    print(f'→ Chênh lệch LỚN HƠN độ lệch chuẩn ⇒ {nhat.mo_hinh} thực sự tốt hơn')

UNG_VIEN = xep_hang.loc[~xep_hang.mo_hinh.isin(['Dummy (mean)', 'Linear (baseline)']), 'mo_hinh'].head(3).tolist()
print(f'\n2–3 mô hình đưa sang bước tinh chỉnh: {UNG_VIEN}')

Hạng 1: XGBoost = 0.3071 ± 0.0015 (39.6s)
Hạng 2: Random Forest = 0.3297 ± 0.0023 (227.5s)
Chênh lệch = 0.0226 | std của hạng 1 = 0.0015
→ Chênh lệch LỚN HƠN độ lệch chuẩn ⇒ XGBoost thực sự tốt hơn

2–3 mô hình đưa sang bước tinh chỉnh: ['XGBoost', 'Random Forest', 'MLP']


## Cell 13 – Đánh giá trên final holdout — **một lần duy nhất**
Mô hình được chọn dựa trên **điểm CV** ở Cell 12, không dựa trên điểm holdout. Bảng dưới báo cáo cả bộ mô hình để có cơ sở so sánh, nhưng **không đổi lựa chọn sau khi nhìn thấy điểm holdout** (nguyên tắc 3).

In [16]:
holdout_rows = []
for name, model in models.items():
    pipe = model if isinstance(model, Pipeline) else Pipeline([('pre', pre), ('model', model)])
    t0 = time.time()
    pipe.fit(X_dev, y_dev)
    fit_time = time.time() - t0
    pred_log = pipe.predict(X_hold)
    holdout_rows.append({
        'mo_hinh': name,
        'Holdout_RMSLE': rmsle(y_hold, pred_log),               # thang log
        'Holdout_MAE_seconds': mae_seconds(y_hold, pred_log),   # thang giây
        'Holdout_RMSE_seconds': rmse_seconds(y_hold, pred_log), # thang giây
        'Holdout_R2': r2_score(y_hold, pred_log),               # thang log
        'Training_time': fit_time,
    })
    print(f'{name:<18} RMSLE = {holdout_rows[-1]["Holdout_RMSLE"]:.4f} '
          f'| MAE = {holdout_rows[-1]["Holdout_MAE_seconds"]:,.1f} giây | fit {fit_time:,.1f}s')

test_table = pd.DataFrame(holdout_rows)

Dummy (mean)       RMSLE = 0.7217 | MAE = 440.0 giây | fit 0.4s
Linear (baseline)  RMSLE = 0.4078 | MAE = 237.9 giây | fit 0.6s
Ridge              RMSLE = 0.4079 | MAE = 238.0 giây | fit 0.5s
Random Forest      RMSLE = 0.3299 | MAE = 183.7 giây | fit 56.9s
XGBoost            RMSLE = 0.3069 | MAE = 167.9 giây | fit 8.1s
MLP                RMSLE = 0.3437 | MAE = 193.9 giây | fit 28.3s


## Cell 14 – Bảng kết quả cuối (mô hình tốt nhất **in đậm**)
**Cột nào thuộc thang nào:**
- **Thang log (`log1p`):** `CV_RMSLE_*`, `CV_MAE_log_mean`, `CV_R2_mean`, `Holdout_RMSLE`, `Holdout_R2`
- **Thang giây (sau `expm1`):** `CV_MAE_seconds_mean`, `CV_RMSE_seconds_mean`, `Holdout_MAE_seconds`, `Holdout_RMSE_seconds`

In [17]:
bang = (cv_table.rename(columns={
            'mo_hinh': 'Model',
            'rmsle_mean': 'CV_RMSLE_mean', 'rmsle_std': 'CV_RMSLE_std', 'rmsle_train': 'CV_RMSLE_train',
            'mae_log_mean': 'CV_MAE_log_mean', 'mae_seconds_mean': 'CV_MAE_seconds_mean',
            'rmse_seconds_mean': 'CV_RMSE_seconds_mean', 'r2_mean': 'CV_R2_mean'})
          [['Model', 'CV_RMSLE_mean', 'CV_RMSLE_std', 'CV_RMSLE_train', 'CV_MAE_log_mean',
            'CV_MAE_seconds_mean', 'CV_RMSE_seconds_mean', 'CV_R2_mean']]
          .merge(test_table.rename(columns={'mo_hinh': 'Model'}), on='Model')
          .sort_values('CV_RMSLE_mean').reset_index(drop=True))

i_best = int(bang['CV_RMSLE_mean'].idxmin())
BEST_NAME = bang.loc[i_best, 'Model']

def in_dam(row):
    return ['font-weight: bold; background-color: #e8f5e9' if row.name == i_best else '' for _ in row]

display(bang.style.apply(in_dam, axis=1).format({
    'CV_RMSLE_mean': '{:.4f}', 'CV_RMSLE_std': '{:.4f}', 'CV_RMSLE_train': '{:.4f}',
    'CV_MAE_log_mean': '{:.4f}', 'CV_MAE_seconds_mean': '{:,.1f}', 'CV_RMSE_seconds_mean': '{:,.1f}',
    'CV_R2_mean': '{:.4f}', 'Holdout_RMSLE': '{:.4f}', 'Holdout_MAE_seconds': '{:,.1f}',
    'Holdout_RMSE_seconds': '{:,.1f}', 'Holdout_R2': '{:.4f}', 'Training_time': '{:,.1f}',
}).hide(axis='index'))

cv_table.to_csv('models/cv_results.csv', index=False, encoding='utf-8-sig')
test_table.to_csv('models/test_results.csv', index=False, encoding='utf-8-sig')
bang.to_csv(f'models/baseline_comparison_{MODE}.csv', index=False, encoding='utf-8-sig')
print(f'Đã lưu: models/cv_results.csv · models/test_results.csv · models/baseline_comparison_{MODE}.csv')
print(f'MÔ HÌNH TỐT NHẤT THEO CV: {BEST_NAME}')

Model,CV_RMSLE_mean,CV_RMSLE_std,CV_RMSLE_train,CV_MAE_log_mean,CV_MAE_seconds_mean,CV_RMSE_seconds_mean,CV_R2_mean,Holdout_RMSLE,Holdout_MAE_seconds,Holdout_RMSE_seconds,Holdout_R2,Training_time
XGBoost,0.3071,0.0015,0.2481,0.2236,167.9,295.7,0.8217,0.3069,167.9,292.1,0.8192,8.1
Random Forest,0.3297,0.0023,0.2643,0.2436,183.1,316.1,0.7945,0.3299,183.7,314.0,0.7910,56.9
MLP,0.3459,0.0019,0.3414,0.2564,195.7,342.2,0.7738,0.3437,193.9,328.7,0.7732,28.3
Linear (baseline),0.4035,0.0026,0.4034,0.3078,235.2,385.2,0.6923,0.4078,237.9,386.1,0.6806,0.6
Ridge,0.4046,0.0027,0.4045,0.3088,235.8,385.8,0.6906,0.4079,238.0,386.2,0.6805,0.5
Dummy (mean),0.7274,0.0016,0.7274,0.5815,443.0,689.9,-0.0000,0.7217,440.0,684.5,-0.0000,0.4


Đã lưu: models/cv_results.csv · models/test_results.csv · models/baseline_comparison_fast.csv
MÔ HÌNH TỐT NHẤT THEO CV: XGBoost


## Cell 15 – So sánh với baseline
**Hai tầng baseline** theo nguyên tắc 4:
1. **Baseline nội bộ** – Linear Regression, cùng dữ liệu và cùng cách đánh giá ⇒ **so sánh có giá trị**.
2. **Baseline bài báo** – các con số công bố trên cùng dataset Kaggle. Đây chỉ là **mốc tham khảo**: các bài báo đánh giá trên tập test của cuộc thi (giữ nguyên ngoại lai), còn nhóm đánh giá trên holdout đã qua cleaning ⇒ **khác điều kiện**.

In [18]:
lin = bang.loc[bang.Model == 'Linear (baseline)'].iloc[0]
dum = bang.loc[bang.Model == 'Dummy (mean)'].iloc[0]
best = bang.loc[i_best]

cai_thien_lin = (lin.CV_RMSLE_mean - best.CV_RMSLE_mean) / lin.CV_RMSLE_mean
cai_thien_dum = (dum.CV_RMSLE_mean - best.CV_RMSLE_mean) / dum.CV_RMSLE_mean

so_sanh = pd.DataFrame([
    {'Nguồn': 'Poongodi và cộng sự (2022) – XGBoost + K-Means so với MLP',
     'Dữ liệu': 'Cùng dataset Kaggle', 'Cách đánh giá': 'Tập test của cuộc thi', 'Metric': 'RMSLE',
     'Giá trị': '— (bài báo không công bố con số so sánh được)',
     'Cùng điều kiện?': 'Không – chỉ dùng làm tham chiếu định tính'},
    {'Nguồn': 'Nghiên cứu công bố trên Kaggle (XGBoost + MiniBatch K-Means + OSRM)',
     'Dữ liệu': 'Cùng dataset + dữ liệu lộ trình OSRM',
     'Cách đánh giá': 'Tập test của cuộc thi', 'Metric': 'RMSLE', 'Giá trị': 0.44076,
     'Cùng điều kiện?': 'Không – dùng thêm dữ liệu ngoài'},
    {'Nguồn': 'Baseline nội bộ – Linear Regression', 'Dữ liệu': 'Dữ liệu nhóm (đã cleaning)',
     'Cách đánh giá': '5-fold CV trên 80%', 'Metric': 'RMSLE', 'Giá trị': round(lin.CV_RMSLE_mean, 4),
     'Cùng điều kiện?': 'Có'},
    {'Nguồn': f'Mô hình tốt nhất – {BEST_NAME}', 'Dữ liệu': 'Dữ liệu nhóm (đã cleaning)',
     'Cách đánh giá': '5-fold CV trên 80%', 'Metric': 'RMSLE', 'Giá trị': round(best.CV_RMSLE_mean, 4),
     'Cùng điều kiện?': 'Có'},
])
display(so_sanh)

print(f'Cải thiện so với baseline nội bộ (Linear): {cai_thien_lin:.1%}')
print(f'Cải thiện so với mốc tầm thường (Dummy)  : {cai_thien_dum:.1%}')
print('\nLưu ý khi viết Discussion: con số của bài báo tính trên tập test cuộc thi (còn nguyên ngoại lai)')
print('và có dùng thêm dữ liệu lộ trình OSRM, nên KHÔNG so trực tiếp được với RMSLE holdout của nhóm.')
so_sanh.to_csv('models/baseline_papers.csv', index=False, encoding='utf-8-sig')

,Nguồn,Dữ liệu,Cách đánh giá,Metric,Giá trị,Cùng điều kiện?
0,Poongodi và cộng sự (2022) – XGBoost + K-Means...,Cùng dataset Kaggle,Tập test của cuộc thi,RMSLE,— (bài báo không công bố con số so sánh được),Không – chỉ dùng làm tham chiếu định tính
1,Nghiên cứu công bố trên Kaggle (XGBoost + Mini...,Cùng dataset + dữ liệu lộ trình OSRM,Tập test của cuộc thi,RMSLE,0.44076,Không – dùng thêm dữ liệu ngoài
2,Baseline nội bộ – Linear Regression,Dữ liệu nhóm (đã cleaning),5-fold CV trên 80%,RMSLE,0.4035,Có
3,Mô hình tốt nhất – XGBoost,Dữ liệu nhóm (đã cleaning),5-fold CV trên 80%,RMSLE,0.3071,Có


Cải thiện so với baseline nội bộ (Linear): 23.9%
Cải thiện so với mốc tầm thường (Dummy)  : 57.8%

Lưu ý khi viết Discussion: con số của bài báo tính trên tập test cuộc thi (còn nguyên ngoại lai)
và có dùng thêm dữ liệu lộ trình OSRM, nên KHÔNG so trực tiếp được với RMSLE holdout của nhóm.


## Cell 16 – Kết luận bước này & việc tiếp theo

In [19]:
print('=' * 80)
print(f'MODE = {MODE} | {len(df):,} dòng sau cleaning (dev {len(X_dev):,} / holdout {len(X_hold):,}) '
      f'| {len(FEATURES)} đặc trưng | {len(models)} mô hình')
print('=' * 80)
print(f'MÔ HÌNH TỐT NHẤT (theo CV): {BEST_NAME}')
print(f'  CV RMSLE      : {best.CV_RMSLE_mean:.4f} ± {best.CV_RMSLE_std:.4f} (thang log)')
print(f'  Holdout RMSLE : {best.Holdout_RMSLE:.4f} (thang log)')
print(f'  Holdout MAE   : {best.Holdout_MAE_seconds:,.1f} giây | RMSE: {best.Holdout_RMSE_seconds:,.1f} giây')
print(f'  Chênh CV vs Holdout: {abs(best.CV_RMSLE_mean - best.Holdout_RMSLE):.4f} '
      f'(chênh nhỏ ⇒ kết quả CV đáng tin)')
print(f'  Dấu hiệu quá khớp  : train {best.CV_RMSLE_train:.4f} vs CV {best.CV_RMSLE_mean:.4f} '
      f'(chênh {best.CV_RMSLE_mean - best.CV_RMSLE_train:.4f})')
print(f'\nTổng thời gian CV 5 mô hình: {cv_table.giay.sum():,.1f}s '
      f'| Tổng thời gian fit trên 80%: {bang.Training_time.sum():,.1f}s')
print(f'\nĐưa sang bước tinh chỉnh: {UNG_VIEN}')

MODE = fast | 200,000 dòng sau cleaning (dev 160,000 / holdout 40,000) | 30 đặc trưng | 6 mô hình
MÔ HÌNH TỐT NHẤT (theo CV): XGBoost
  CV RMSLE      : 0.3071 ± 0.0015 (thang log)
  Holdout RMSLE : 0.3069 (thang log)
  Holdout MAE   : 167.9 giây | RMSE: 292.1 giây
  Chênh CV vs Holdout: 0.0002 (chênh nhỏ ⇒ kết quả CV đáng tin)
  Dấu hiệu quá khớp  : train 0.2481 vs CV 0.3071 (chênh 0.0590)

Tổng thời gian CV 5 mô hình: 389.8s | Tổng thời gian fit trên 80%: 94.8s

Đưa sang bước tinh chỉnh: ['XGBoost', 'Random Forest', 'MLP']


### Giới hạn của bước này (ghi vào báo cáo)
1. **Chưa tinh chỉnh siêu tham số** – mọi mô hình dùng cấu hình mặc định hợp lý, nên thứ hạng có thể đổi sau khi tinh chỉnh.
2. **Chạy trên mẫu con**, không phải toàn bộ dữ liệu sau cleaning, để phù hợp giới hạn thời gian của Kaggle. Cần ghi rõ `MODE` và cỡ mẫu khi báo cáo số liệu.
3. **Holdout đã qua cleaning** nên "dễ" hơn tập test thật của cuộc thi ⇒ không so trực tiếp với leaderboard.
4. **Chưa dùng PCA/K-Means và dữ liệu ngoài** (lộ trình OSRM, thời tiết) mà các nghiên cứu công bố có dùng.
5. kNN chỉ chạy khi bật `USE_KNN` vì chi phí tính toán rất lớn trên dữ liệu này.

### Bước tiếp theo (notebook sau)
Theo Mục 7–11 của hướng dẫn: tinh chỉnh 2–3 mô hình tốt nhất bằng `RandomizedSearchCV` cùng `cv` và cùng metric chính → ghi `models/tuning_log.csv` (cấu hình và điểm CV trước/sau) → chốt mô hình → giải thích mô hình (hệ số, permutation importance, SHAP) và phân tích sai số theo nhóm → vẽ Dự đoán vs Thực tế, phần dư → lưu `models/best_model.joblib` cùng `models/model_meta.json`.

In [20]:
import os

# 1) Dataset đã gắn vào notebook chưa?
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

# 2) Biến dữ liệu nào đang có trong bộ nhớ?
for ten in ['raw', 'train', 'df', 'X_all']:
    if ten in globals():
        print(f'{ten}: {globals()[ten].shape}')


/kaggle/input/competitions/nyc-taxi-trip-duration/train.zip
/kaggle/input/competitions/nyc-taxi-trip-duration/test.zip
/kaggle/input/competitions/nyc-taxi-trip-duration/sample_submission.zip
raw: (1458644, 11)
df: (200000, 36)


In [21]:
# =============================================================================
#  CELL BỔ SUNG — sinh bảng audit chất lượng dữ liệu
#
#  ĐẶT Ở ĐÂU:  cuối notebook **Phần 4 (Modeling)**.
#  Vì sao Phần 4: cell 6 của notebook đó đã nạp sẵn biến `raw`
#  (DataFrame gốc 1.458.644 dòng, chưa làm sạch) — đúng thứ cell này cần.
#  Notebook Phần 1 đặt tên biến khác, nếu muốn chạy ở đó thì sửa dòng
#  `DU_LIEU_GOC = raw` bên dưới cho khớp tên biến của bạn.
#
#  KẾT QUẢ:  models/audit.csv
#  Tải file đó về, bỏ vào thư mục models/ cạnh make_tables.py, chạy
#      python make_tables.py
#  rồi upload tables/table_audit.tex lên Overleaf. Khung vàng sẽ biến mất.
# =============================================================================

import os
import pandas as pd

DU_LIEU_GOC = raw          # <-- đổi tên biến ở đây nếu notebook của bạn khác

os.makedirs('models', exist_ok=True)


def dem_iqr(s):
    """Số dòng bị luật IQR gắn cờ ngoại lai."""
    s = pd.to_numeric(s, errors='coerce').dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


def dem_zscore(s):
    """Số dòng có |z| > 3."""
    s = pd.to_numeric(s, errors='coerce').dropna()
    sd = s.std()
    if sd == 0 or pd.isna(sd):
        return 0
    return int((((s - s.mean()) / sd).abs() > 3).sum())


# Luật theo miền — PHẢI khớp với các bất đẳng thức (3)–(7) trong bài báo.
# Nếu bạn đã đổi ngưỡng trong notebook thì sửa ở đây cho khớp.
LUAT_MIEN = {
    'trip_duration':     lambda d: (d.trip_duration < 60) | (d.trip_duration > 21600),
    'passenger_count':   lambda d: (d.passenger_count < 1) | (d.passenger_count > 6),
    'pickup_latitude':   lambda d: (d.pickup_latitude < 40.49) | (d.pickup_latitude > 40.92),
    'dropoff_latitude':  lambda d: (d.dropoff_latitude < 40.49) | (d.dropoff_latitude > 40.92),
    'pickup_longitude':  lambda d: (d.pickup_longitude < -74.27) | (d.pickup_longitude > -73.68),
    'dropoff_longitude': lambda d: (d.dropoff_longitude < -74.27) | (d.dropoff_longitude > -73.68),
}

# Giá trị thiếu "trá hình": hợp lệ về kiểu nhưng vô lý về nghiệp vụ.
THIEU_TRA_HINH = {
    'passenger_count': lambda s: int((s == 0).sum()),
}

dong = []
for cot in DU_LIEU_GOC.columns:
    s = DU_LIEU_GOC[cot]
    la_so = pd.api.types.is_numeric_dtype(s) and cot != 'id'
    n_domain = int(LUAT_MIEN[cot](DU_LIEU_GOC).sum()) if cot in LUAT_MIEN else 0

    dong.append({
        # Tên cột để NGUYÊN BẢN — make_tables.py tự thoát ký tự LaTeX.
        'cot':      cot,
        'kieu':     'numeric' if la_so else 'categorical',
        'thieu':    int(s.isna().sum()),
        'thieu_an': THIEU_TRA_HINH[cot](s) if cot in THIEU_TRA_HINH else 0,
        'iqr':      dem_iqr(s) if la_so else 0,
        'zscore':   dem_zscore(s) if la_so else 0,
        'domain':   n_domain,
    })

audit = pd.DataFrame(dong)
audit.to_csv('models/audit.csv', index=False)

print('=== BẢNG AUDIT ===')
print(audit.to_string(index=False))
print('\n→ đã ghi models/audit.csv')

# --- Con số chứng minh câu "lệch hai bậc độ lớn" ----------------------------
pc = audit[audit.cot == 'passenger_count']
if len(pc):
    pc = pc.iloc[0]
    print(f'\npassenger_count:')
    print(f'  IQR gắn cờ        : {pc.iqr:,} dòng')
    print(f'  Luật nghiệp vụ    : {pc.domain:,} dòng')
    if pc.domain > 0:
        ty_le = pc.iqr / pc.domain
        print(f'  → lệch {ty_le:,.0f} lần')
        print(f'  → nếu ≥ 100 thì câu "two orders of magnitude" trong bài là ĐÚNG')
        print(f'  → nếu < 100 thì sửa câu đó thành "a factor of {ty_le:,.0f}"')
    print(f'\n  Số dòng passenger_count = 0 (thiếu trá hình): {pc.thieu_an:,}')

# --- Nhắc việc còn lại ------------------------------------------------------
print('\n--- CÒN PHẢI ĐIỀN TAY TRONG main.tex ---')
print(r'  \NRulesPass   : trong 16 luật nhất quán ở notebook Phần 1, bao nhiêu luật ĐẠT')
print(r'  \NTuneSample  : số dòng truyền vào RandomizedSearchCV (notebook Phần 5)')
print(r'  \NTuneIter    : tham số n_iter của RandomizedSearchCV')
print(r'  \RepoURL      : link GitHub của nhóm')

# --- Sai số tuyệt đối lớn nhất, cho macro \MaxAbsError -----------------------
# Bỏ dấu # ở dưới nếu chạy trong notebook Phần 5, nơi đã có y_true/y_pred.
# sai_so = (y_true_seconds - y_pred_seconds).abs()
# print(f'\n\\MaxAbsError = {sai_so.max()/3600:.1f}~h  ({sai_so.max():,.0f} giây)')


=== BẢNG AUDIT ===
               cot        kieu  thieu  thieu_an    iqr  zscore  domain
                id categorical      0         0      0       0       0
         vendor_id     numeric      0         0      0       0       0
   pickup_datetime categorical      0         0      0       0       0
  dropoff_datetime categorical      0         0      0       0       0
   passenger_count     numeric      0        60 154830   48338      65
  pickup_longitude     numeric      0         0  84322     367     186
   pickup_latitude     numeric      0         0  52743   33563     170
 dropoff_longitude     numeric      0         0  77969    2474     695
  dropoff_latitude     numeric      0         0  71990   17298     632
store_and_fwd_flag categorical      0         0      0       0       0
     trip_duration     numeric      0         0  74220    2073   10656

→ đã ghi models/audit.csv

passenger_count:
  IQR gắn cờ        : 154,830 dòng
  Luật nghiệp vụ    : 65 dòng
  → lệch 2,382 lần


In [22]:
# =============================================================================
#  LẤY SỐ CHO HAI MỤC B4 VÀ B6 CỦA PHẢN BIỆN v2
#
#  ĐẶT Ở ĐÂU: cuối notebook PHẦN 4 (Modeling) — nơi có sẵn biến `raw`.
#  Chạy xong, copy nguyên phần in ra gửi lại, mình điền vào bài.
# =============================================================================

import numpy as np
import pandas as pd

d = raw.copy()

# --- khoảng cách haversine (km) ---------------------------------------------
R = 6371.0
p1 = np.radians(d.pickup_latitude.values)
p2 = np.radians(d.dropoff_latitude.values)
dphi = p2 - p1
dlam = np.radians(d.dropoff_longitude.values - d.pickup_longitude.values)
a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
d['dist_km'] = 2 * R * np.arcsin(np.sqrt(a))
d['speed_kmh'] = d.dist_km / (d.trip_duration / 3600.0)


# =============================================================================
#  B4 — hệ số tương quan khoảng cách ↔ thời gian
#  Bài đang chỉ ghi "low" và "far higher", reviewer đòi con số cụ thể.
# =============================================================================
print('=== B4: TƯƠNG QUAN KHOẢNG CÁCH ↔ THỜI GIAN ===')

pearson_tho = d.dist_km.corr(d.trip_duration, method='pearson')
spearman_tho = d.dist_km.corr(d.trip_duration, method='spearman')

# sau khi làm sạch + lấy log, đúng như bài mô tả
sach = d[
    d.trip_duration.between(60, 21600)
    & d.passenger_count.between(1, 6)
    & d.pickup_latitude.between(40.49, 40.92)
    & d.dropoff_latitude.between(40.49, 40.92)
    & d.pickup_longitude.between(-74.27, -73.68)
    & d.dropoff_longitude.between(-74.27, -73.68)
    & (d.dist_km >= 0.05)
    & (d.speed_kmh <= 100)
].copy()
pearson_sach = np.log1p(sach.dist_km).corr(np.log1p(sach.trip_duration))

print(f'  Pearson  trên dữ liệu thô         : {pearson_tho:.3f}')
print(f'  Spearman trên dữ liệu thô         : {spearman_tho:.3f}')
print(f'  Pearson  sau làm sạch + log       : {pearson_sach:.3f}')
print(f'  (còn lại {len(sach):,} dòng sau làm sạch)')
print('\n>>> GỬI MÌNH BA SỐ TRÊN')


# =============================================================================
#  B6 — bảng lọc tuần tự (thác nước)
#  reviewer: "các luật chồng lên nhau nên người đọc không cộng ra được 1,23%"
# =============================================================================
print('\n\n=== B6: LỌC TUẦN TỰ ===')

con = d.copy()
buoc = []


def loc(ten, mask_giu):
    """Áp một bước lọc, ghi lại số dòng bị loại và số còn lại."""
    global con
    truoc = len(con)
    con = con[mask_giu(con)]
    buoc.append((ten, truoc - len(con), len(con)))


buoc.append(('Raw training data', 0, len(con)))
loc('Exact duplicates',        lambda x: ~x.drop(columns=['id', 'dist_km', 'speed_kmh']).duplicated())
loc('Duration bounds',         lambda x: x.trip_duration.between(60, 21600))
loc('Passenger bounds',        lambda x: x.passenger_count.between(1, 6))
loc('Latitude bounds',         lambda x: x.pickup_latitude.between(40.49, 40.92)
                                         & x.dropoff_latitude.between(40.49, 40.92))
loc('Longitude bounds',        lambda x: x.pickup_longitude.between(-74.27, -73.68)
                                         & x.dropoff_longitude.between(-74.27, -73.68))
loc('Minimum distance',        lambda x: x.dist_km >= 0.05)
loc('Maximum speed',           lambda x: x.speed_kmh <= 100)

print(f'{"Step":<22}{"Removed":>10}{"Remaining":>12}')
for ten, bo, lai in buoc:
    print(f'{ten:<22}{bo:>10,}{lai:>12,}')

tong_bo = len(d) - len(con)
print(f'\nTổng loại: {tong_bo:,} dòng = {tong_bo/len(d)*100:.2f}%')
print(f'Còn lại  : {len(con):,} dòng')
print('Bài đang ghi: loại 17.893 dòng (1,23%), còn 1.440.751')

# --- in sẵn thân bảng LaTeX -------------------------------------------------
print('\n========== COPY TỪ ĐÂY ==========')
for ten, bo, lai in buoc:
    bo_txt = '---' if bo == 0 else f'{bo:,}'.replace(',', '{,}')
    lai_txt = f'{lai:,}'.replace(',', '{,}')
    print(f'{ten:<22} & {bo_txt:>12} & ${lai_txt}$ \\\\')
print('========== TỚI ĐÂY ==========')

=== B4: TƯƠNG QUAN KHOẢNG CÁCH ↔ THỜI GIAN ===
  Pearson  trên dữ liệu thô         : 0.095
  Spearman trên dữ liệu thô         : 0.794
  Pearson  sau làm sạch + log       : 0.783
  (còn lại 1,440,756 dòng sau làm sạch)

>>> GỬI MÌNH BA SỐ TRÊN


=== B6: LỌC TUẦN TỰ ===
Step                     Removed   Remaining
Raw training data              0   1,458,644
Exact duplicates               7   1,458,637
Duration bounds           10,654   1,447,983
Passenger bounds              19   1,447,964
Latitude bounds              556   1,447,408
Longitude bounds             488   1,446,920
Minimum distance           6,136   1,440,784
Maximum speed                 33   1,440,751

Tổng loại: 17,893 dòng = 1.23%
Còn lại  : 1,440,751 dòng
Bài đang ghi: loại 17.893 dòng (1,23%), còn 1.440.751

========== COPY TỪ ĐÂY ==========
Raw training data      &          --- & $1{,}458{,}644$ \\
Exact duplicates       &            7 & $1{,}458{,}637$ \\
Duration bounds        &     10{,}654 & $1{,}447{,}983$ \\
P

In [23]:
# =============================================================================
#  THÍ NGHIỆM CHIA THEO THỜI GIAN  (mục B7 của phản biện v2)
#
#  ĐẶT Ở ĐÂU: cuối notebook PHẦN 4 (Modeling).
#  Cần có sẵn: df, X, y, FEATURES, NUM, CAT, pre, SEED
#  (tất cả đều do notebook Phần 4 tạo ra, không phải sửa gì phía trên).
#
#  THỜI GIAN CHẠY: ba lần fit XGBoost trên ~1,44 triệu dòng.
#  Đặt MAU = 300_000 để chạy thử nhanh, MAU = None để chạy thật.
# =============================================================================

import time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

MAU = None          # None = dùng toàn bộ dữ liệu đã làm sạch

# =============================================================================
#  VÌ SAO PHẢI CÓ BA NHÁNH, KHÔNG PHẢI HAI
# =============================================================================
#  `month` và `day_of_year` NẰM TRONG danh sách 30 đặc trưng.
#
#  Nếu train tháng 1–5 rồi test tháng 6, mô hình chưa từng thấy month = 6,
#  và day_of_year của tập test (152–182) hoàn toàn tách rời tập train (1–151).
#  Cây quyết định gặp giá trị ngoài khoảng đã học sẽ dồn hết về một nhánh.
#
#  Nên nếu chỉ so "ngẫu nhiên" với "theo thời gian", phần lớn chênh lệch đo
#  được là do HAI CỘT NÀY, không phải do trôi dữ liệu thật. Đó là lý do cần
#  nhánh thứ ba: bỏ hai cột đó đi rồi mới đo.
# =============================================================================

DAC_TRUNG_THEO_LICH = ['month', 'day_of_year']

CAU_HINH_XGB = dict(n_estimators=500, learning_rate=0.05, max_depth=8,
                    min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                    reg_lambda=1.0, tree_method='hist', n_jobs=-1,
                    random_state=SEED)


def dung_pipeline(cot_so, cot_phanloai):
    """Dựng lại pipeline y hệt notebook, nhưng cho một tập cột tuỳ chọn."""
    num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('sc', StandardScaler())])
    cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                         ('oh', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                              min_frequency=20, sparse_output=False))])
    tien_xu_ly = ColumnTransformer([('num', num_pipe, cot_so),
                                    ('cat', cat_pipe, cot_phanloai)])
    return Pipeline([('pre', tien_xu_ly),
                     ('model', XGBRegressor(**CAU_HINH_XGB))])


def do_luong(ten, Xtr, ytr, Xte, yte, cot_so, cot_phanloai):
    """Fit rồi trả về RMSLE và MAE (giây) trên tập test."""
    t0 = time.time()
    mo_hinh = dung_pipeline(cot_so, cot_phanloai).fit(Xtr, ytr)
    du_doan = mo_hinh.predict(Xte)

    # y đã là log1p(trip_duration), nên RMSLE = RMSE ngay trên thang này
    rmsle = float(np.sqrt(np.mean((np.asarray(yte) - du_doan) ** 2)))
    mae_giay = mean_absolute_error(np.expm1(np.asarray(yte)),
                                   np.clip(np.expm1(du_doan), 0, None))

    print(f'  {ten:<42} RMSLE {rmsle:.4f} | MAE {mae_giay:7,.0f}s '
          f'| train {len(Xtr):,} test {len(Xte):,} | {time.time()-t0:,.0f}s')
    return rmsle, mae_giay, len(Xtr), len(Xte)


# =============================================================================
#  CHUẨN BỊ
# =============================================================================
d = df if MAU is None else df.sample(MAU, random_state=SEED)
Xd, yd = d[FEATURES].astype('float32'), d[TARGET].astype('float32')
thang = d['month'].astype(int)

mask_tr = (thang <= 5).values
mask_te = (thang == 6).values

print('=== PHÂN BỐ THEO THÁNG ===')
print(thang.value_counts().sort_index().to_string())
ty_le_test = mask_te.mean()
print(f'\nTháng 1–5 : {mask_tr.sum():,} dòng ({1-ty_le_test:.1%})')
print(f'Tháng 6   : {mask_te.sum():,} dòng ({ty_le_test:.1%})')

NUM_bo = [c for c in NUM if c not in DAC_TRUNG_THEO_LICH]
print(f'\nNhánh C bỏ {DAC_TRUNG_THEO_LICH} → còn {len(NUM_bo)+len(CAT)} đặc trưng')

ket_qua = {}
print('\n=== BA NHÁNH ===')

# --- A: chia ngẫu nhiên, cùng tỉ lệ test (đối chứng) ------------------------
Xtr_a, Xte_a, ytr_a, yte_a = train_test_split(
    Xd, yd, test_size=ty_le_test, random_state=SEED)
ket_qua['A'] = do_luong('A. Ngẫu nhiên, đủ 30 đặc trưng',
                        Xtr_a, ytr_a, Xte_a, yte_a, NUM, CAT)

# --- B: chia theo thời gian, đủ đặc trưng (có lẫn ảnh hưởng month) ----------
ket_qua['B'] = do_luong('B. Theo thời gian, đủ 30 đặc trưng',
                        Xd[mask_tr], yd[mask_tr], Xd[mask_te], yd[mask_te], NUM, CAT)

# --- C: chia theo thời gian, bỏ month + day_of_year (phép đo thật) ---------
ket_qua['C'] = do_luong('C. Theo thời gian, bỏ month/day_of_year',
                        Xd[mask_tr], yd[mask_tr], Xd[mask_te], yd[mask_te], NUM_bo, CAT)

# =============================================================================
#  ĐỌC KẾT QUẢ
# =============================================================================
rA, rB, rC = ket_qua['A'][0], ket_qua['B'][0], ket_qua['C'][0]

print('\n=== DIỄN GIẢI ===')
print(f'  A (ngẫu nhiên)              : {rA:.4f}')
print(f'  B (thời gian, đủ đặc trưng) : {rB:.4f}   lệch A {(rB-rA)/rA:+.1%}')
print(f'  C (thời gian, bỏ lịch)      : {rC:.4f}   lệch A {(rC-rA)/rA:+.1%}')
print()
print(f'  Do hai cột month/day_of_year : {(rB-rC)/rA:+.1%} (B trừ C)')
print(f'  Do trôi dữ liệu thật         : {(rC-rA)/rA:+.1%} (C trừ A)  <-- CON SỐ CẦN BÁO CÁO')
print()
if rC > rA:
    print(f'  => Chia ngẫu nhiên LẠC QUAN hơn {(rC-rA)/rA:.1%} so với dự đoán về tương lai.')
else:
    print('  => Không thấy lạc quan đáng kể. Cũng là một kết quả, báo cáo trung thực.')

# =============================================================================
#  THÂN BẢNG LaTeX — copy dán vào bài
# =============================================================================
print('\n========== COPY TỪ ĐÂY ==========')
nhan = {'A': r'Random, all \NFeatures{} features',
        'B': r'Chronological, all \NFeatures{} features',
        'C': r'Chronological, calendar features removed'}
for k in ('A', 'B', 'C'):
    r, mae, ntr, nte = ket_qua[k]
    d_txt = '---' if k == 'A' else f'{(r-rA)/rA:+.1%}'.replace('%', r'\%')
    print(f'{nhan[k]:<46} & {r:.4f} & {mae:,.0f} & {d_txt} \\\\'.replace(
        f'{mae:,.0f}', f'{mae:,.0f}'.replace(',', '{,}')))
print('========== TỚI ĐÂY ==========')

pd.DataFrame(
    [(k, *ket_qua[k]) for k in ('A', 'B', 'C')],
    columns=['nhanh', 'rmsle', 'mae_giay', 'n_train', 'n_test']
).to_csv('models/temporal_split.csv', index=False)
print('\n→ đã ghi models/temporal_split.csv')

=== PHÂN BỐ THEO THÁNG ===
month
1    31498
2    32664
3    35316
4    34427
5    34152
6    31943

Tháng 1–5 : 168,057 dòng (84.0%)
Tháng 6   : 31,943 dòng (16.0%)

Nhánh C bỏ ['month', 'day_of_year'] → còn 28 đặc trưng

=== BA NHÁNH ===
  A. Ngẫu nhiên, đủ 30 đặc trưng             RMSLE 0.3054 | MAE     167s | train 168,057 test 31,943 | 8s
  B. Theo thời gian, đủ 30 đặc trưng         RMSLE 0.3450 | MAE     204s | train 168,057 test 31,943 | 8s
  C. Theo thời gian, bỏ month/day_of_year    RMSLE 0.3227 | MAE     190s | train 168,057 test 31,943 | 8s

=== DIỄN GIẢI ===
  A (ngẫu nhiên)              : 0.3054
  B (thời gian, đủ đặc trưng) : 0.3450   lệch A +12.9%
  C (thời gian, bỏ lịch)      : 0.3227   lệch A +5.7%

  Do hai cột month/day_of_year : +7.3% (B trừ C)
  Do trôi dữ liệu thật         : +5.7% (C trừ A)  <-- CON SỐ CẦN BÁO CÁO

  => Chia ngẫu nhiên LẠC QUAN hơn 5.7% so với dự đoán về tương lai.

========== COPY TỪ ĐÂY ==========
Random, all \NFeatures{} features              & 

In [24]:
# =============================================================================
#  HAI THÍ NGHIỆM THEO PHIẾU PHẢN BIỆN C1–C8
#
#  C1 [BẮT BUỘC] — nhánh thứ tư "Random-28" để hoàn thiện thiết kế 2×2.
#      Bảng XII hiện so Chrono-28 với Random-30: hai nhánh khác nhau CÙNG LÚC
#      cách chia lẫn bộ đặc trưng, nên không tách được drift. Thêm nhánh D
#      (ngẫu nhiên, 28 đặc trưng) mới có:
#          drift thuần         = (Chrono-28 − Random-28) / Random-28
#          giá trị 2 cột lịch  = (Random-28 − Random-30) / Random-30
#          lạc quan của bài    = (Chrono-28 − Random-30) / Random-30  ← 5,7%
#      Chi phí: một lần fit trên 200k, khoảng 40 giây.
#
#  C2 [NÊN LÀM] — tách hai hiệu ứng trong con số 40,4%.
#      Cách (ii) của phiếu: chấm mô hình trên 17.893 dòng ĐÃ BỊ LOẠI. Những
#      dòng này chưa bao giờ được dùng để huấn luyện, và nhóm CÓ nhãn của
#      chúng — khác tập test Kaggle. Báo RMSLE theo từng luật lọc cho biết
#      luật nào gây phần lớn chênh lệch.
#
#  ĐẶT Ở ĐÂU: cuối notebook PHẦN 4 (cần df, raw, FEATURES, NUM, CAT, TARGET,
#  SEED). Phần C2 cần thêm FINAL_MODEL và add_features.
# =============================================================================

import time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

MAU = None                 # None = dùng toàn bộ df đã làm sạch, như thí nghiệm gốc
COT_LICH = ['month', 'day_of_year']

CAU_HINH = dict(n_estimators=500, learning_rate=0.05, max_depth=8,
                min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
                reg_lambda=1.0, tree_method='hist', n_jobs=-1,
                random_state=SEED)


def dung(cot_so, cot_pl):
    num = Pipeline([('imp', SimpleImputer(strategy='median')),
                    ('sc', StandardScaler())])
    cat = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                    ('oh', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                         min_frequency=20,
                                         sparse_output=False))])
    return Pipeline([('pre', ColumnTransformer([('num', num, cot_so),
                                                ('cat', cat, cot_pl)])),
                     ('model', XGBRegressor(**CAU_HINH))])


def do(ten, Xtr, ytr, Xte, yte, cot_so, cot_pl):
    t0 = time.time()
    m = dung(cot_so, cot_pl).fit(Xtr, ytr)
    p = m.predict(Xte)
    r = float(np.sqrt(np.mean((np.asarray(yte) - p) ** 2)))
    mae = mean_absolute_error(np.expm1(np.asarray(yte)),
                              np.clip(np.expm1(p), 0, None))
    print(f'  {ten:<44} RMSLE {r:.4f} | MAE {mae:7,.0f}s | {time.time()-t0:,.0f}s')
    return r, mae


# =============================================================================
#  C1 — THIẾT KẾ 2×2 ĐẦY ĐỦ
# =============================================================================
d = df if MAU is None else df.sample(MAU, random_state=SEED)
Xd, yd = d[FEATURES].astype('float32'), d[TARGET].astype('float32')
thang = d['month'].astype(int)
mask_tr, mask_te = (thang <= 5).values, (thang == 6).values
ty_le = mask_te.mean()

NUM_bo = [c for c in NUM if c not in COT_LICH]
print(f'Đủ đặc trưng: {len(NUM)+len(CAT)} | bỏ lịch: {len(NUM_bo)+len(CAT)}\n')
print('=== BỐN NHÁNH ===')

Xa, Xb, ya, yb = train_test_split(Xd, yd, test_size=ty_le, random_state=SEED)
A = do('A. Ngẫu nhiên, 30 đặc trưng', Xa, ya, Xb, yb, NUM, CAT)
D = do('D. Ngẫu nhiên, 28 đặc trưng  (MỚI)', Xa, ya, Xb, yb, NUM_bo, CAT)
B = do('B. Theo thời gian, 30 đặc trưng',
       Xd[mask_tr], yd[mask_tr], Xd[mask_te], yd[mask_te], NUM, CAT)
C = do('C. Theo thời gian, 28 đặc trưng',
       Xd[mask_tr], yd[mask_tr], Xd[mask_te], yd[mask_te], NUM_bo, CAT)

rA, rB, rC, rD = A[0], B[0], C[0], D[0]

print('\n=== BA ĐẠI LƯỢNG TÁCH RA ĐƯỢC ===')
print(f'  (a) Drift thuần          (C so D) : {(rC-rD)/rD:+.1%}')
print(f'  (b) Giá trị 2 cột lịch   (D so A) : {(rD-rA)/rA:+.1%}  '
      f'(dương = bỏ đi thì tệ hơn)')
print(f'  (c) Lạc quan của bài     (C so A) : {(rC-rA)/rA:+.1%}  ← con số 5,7% hiện tại')
print(f'  (d) Hiện tượng giá trị lạ (B so C) : {(rB-rC)/rC:+.1%}')

print('\n=== ĐỌC KẾT QUẢ ===')
if abs((rD - rA) / rA) < 0.005:
    print('  Hai cột lịch gần như vô giá trị ngay cả khi chia ngẫu nhiên.')
    print('  → (a) và (c) gần bằng nhau, câu "drift" trong bài gần đúng.')
else:
    print(f'  Hai cột lịch CÓ giá trị thật khi chia ngẫu nhiên ({(rD-rA)/rA:+.1%}).')
    print('  → con số 5,7% KHÔNG phải drift thuần; drift thuần là (a).')
    print('  → phải sửa lại câu diễn giải ở Mục V-F theo đúng (a) và (c).')

print('\n========== THÂN BẢNG LaTeX ==========')
nhan = [('Random', r'\NFeatures{}', rA, A[1]),
        ('Random', '28', rD, D[1]),
        ('Chronological', r'\NFeatures{}', rB, B[1]),
        ('Chronological', '28', rC, C[1])]
for ten, nf, r, mae in nhan:
    dl = '---' if (ten, nf) == ('Random', r'\NFeatures{}') \
        else f'{(r-rA)/rA:+.1%}'.replace('%', r'\%')
    print(f'{ten:<14} & {nf:<12} & {r:.4f} & {mae:,.0f} & {dl} \\\\'
          .replace(f'{mae:,.0f}', f'{mae:,.0f}'.replace(',', '{,}')))
print('========== HẾT ==========')

pd.DataFrame([{'nhanh': n, 'dac_trung': nf, 'rmsle': r, 'mae': m}
              for n, nf, r, m in nhan]).to_csv('models/temporal_2x2.csv',
                                               index=False)
print('→ models/temporal_2x2.csv')


# =============================================================================
#  C2 — CHẤM TRÊN CÁC DÒNG ĐÃ BỊ LOẠI
# =============================================================================
print('\n\n=== C2: chấm mô hình trên các dòng đã bị loại ===')
thieu = [v for v in ('raw', 'FINAL_MODEL', 'add_features') if v not in globals()]
if thieu:
    print(f'  BỎ QUA: thiếu {thieu}.')
else:
    R = 6371.0
    r0 = raw.copy()
    p1, p2 = np.radians(r0.pickup_latitude.values), np.radians(r0.dropoff_latitude.values)
    dphi, dlam = p2 - p1, np.radians(r0.dropoff_longitude.values - r0.pickup_longitude.values)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlam/2)**2
    r0['_d'] = 2*R*np.arcsin(np.sqrt(a))
    r0['_v'] = r0._d / (r0.trip_duration / 3600.0)

    # nhãn luật ĐẦU TIÊN loại mỗi dòng, theo đúng thứ tự thác nước
    luat = [
        ('Duration bounds',  ~r0.trip_duration.between(60, 21600)),
        ('Passenger bounds', ~r0.passenger_count.between(1, 6)),
        ('Latitude bounds',  ~(r0.pickup_latitude.between(40.49, 40.92)
                               & r0.dropoff_latitude.between(40.49, 40.92))),
        ('Longitude bounds', ~(r0.pickup_longitude.between(-74.27, -73.68)
                               & r0.dropoff_longitude.between(-74.27, -73.68))),
        ('Minimum distance', r0._d < 0.05),
        ('Maximum speed',    r0._v > 100),
    ]
    nhan_luat = pd.Series(index=r0.index, dtype=object)
    for ten, m in luat:
        nhan_luat = nhan_luat.where(nhan_luat.notna() | ~m.values, ten)

    bo = r0[nhan_luat.notna()].copy()
    bo['_luat'] = nhan_luat[nhan_luat.notna()]
    print(f'  {len(bo):,} dòng bị loại (bài ghi 17.893)')

    fe = add_features(bo)
    Xb_ = fe[FEATURES].astype('float32')
    yb_ = np.log1p(bo.trip_duration.values.astype('float32'))
    pb = FINAL_MODEL.predict(Xb_)

    print(f'\n  {"Luật loại":<20}{"n":>8}{"RMSLE":>9}{"MAE (s)":>11}')
    hang = []
    for ten in bo._luat.unique():
        m = (bo._luat == ten).values
        rr = float(np.sqrt(np.mean((yb_[m] - pb[m])**2)))
        mm = float(np.mean(np.abs(np.expm1(yb_[m]) - np.clip(np.expm1(pb[m]), 0, None))))
        hang.append({'luat': ten, 'n': int(m.sum()), 'rmsle': rr, 'mae': mm})
        print(f'  {ten:<20}{m.sum():>8,}{rr:>9.4f}{mm:>11,.0f}')
    rr = float(np.sqrt(np.mean((yb_ - pb)**2)))
    print(f'  {"TẤT CẢ":<20}{len(bo):>8,}{rr:>9.4f}')
    print(f'\n  So sánh: holdout đã làm sạch 0,2990 | Kaggle chưa lọc 0,4199')
    print('  → luật nào có RMSLE cao nhất chính là luật gây phần lớn 40,4%.')

    pd.DataFrame(hang).to_csv('models/removed_rows_score.csv', index=False)
    print('  → models/removed_rows_score.csv')

print('\n\nGỬI MÌNH NGUYÊN PHẦN IN RA. C1 là mục BẮT BUỘC của phiếu.')

Đủ đặc trưng: 30 | bỏ lịch: 28

=== BỐN NHÁNH ===
  A. Ngẫu nhiên, 30 đặc trưng                  RMSLE 0.3054 | MAE     167s | 8s
  D. Ngẫu nhiên, 28 đặc trưng  (MỚI)           RMSLE 0.3136 | MAE     174s | 8s
  B. Theo thời gian, 30 đặc trưng              RMSLE 0.3450 | MAE     204s | 8s
  C. Theo thời gian, 28 đặc trưng              RMSLE 0.3227 | MAE     190s | 8s

=== BA ĐẠI LƯỢNG TÁCH RA ĐƯỢC ===
  (a) Drift thuần          (C so D) : +2.9%
  (b) Giá trị 2 cột lịch   (D so A) : +2.7%  (dương = bỏ đi thì tệ hơn)
  (c) Lạc quan của bài     (C so A) : +5.7%  ← con số 5,7% hiện tại
  (d) Hiện tượng giá trị lạ (B so C) : +6.9%

=== ĐỌC KẾT QUẢ ===
  Hai cột lịch CÓ giá trị thật khi chia ngẫu nhiên (+2.7%).
  → con số 5,7% KHÔNG phải drift thuần; drift thuần là (a).
  → phải sửa lại câu diễn giải ở Mục V-F theo đúng (a) và (c).

========== THÂN BẢNG LaTeX ==========
Random         & \NFeatures{} & 0.3054 & 167 & --- \\
Random         & 28           & 0.3136 & 174 & +2.7\% \\
Chronologica